<!-- project-navigation -->
# 07_data_understandin_pos_cash
Role: Exploration and data preparation.

See [notebook guide](README.md) and [model comparison](../reports/model_comparison.md). Historical code and outputs are retained; run with the project root as working directory.


In [2]:
import pandas as pd

# Chargement de la table
pos_cash = pd.read_csv("data/raw/POS_CASH_balance.csv")

print("========== DIMENSION ==========")
print(pos_cash.shape)

print("\n========== INFORMATIONS ==========")
print(pos_cash.info())

print("\n========== TYPES DES VARIABLES ==========")
print(pos_cash.dtypes.value_counts())

print("\n========== VARIABLES ==========")
print(pos_cash.columns.tolist())

# Valeurs manquantes
missing = pos_cash.isnull().sum()
missing = missing[missing > 0]

missing_table = pd.DataFrame({
    "Nombre_manquant": missing,
    "Pourcentage": round(missing / len(pos_cash) * 100, 2)
}).sort_values(by="Pourcentage", ascending=False)

print("\n========== VALEURS MANQUANTES ==========")
display(missing_table)

print("\n========== APERCU ==========")
display(pos_cash.head())

print("\n========== STATISTIQUES NUMERIQUES ==========")
display(pos_cash.describe())

========== DIMENSION ==========
(10001358, 8)

========== INFORMATIONS ==========
<class 'pandas.DataFrame'>
RangeIndex: 10001358 entries, 0 to 10001357
Data columns (total 8 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   SK_ID_PREV             int64  
 1   SK_ID_CURR             int64  
 2   MONTHS_BALANCE         int64  
 3   CNT_INSTALMENT         float64
 4   CNT_INSTALMENT_FUTURE  float64
 5   NAME_CONTRACT_STATUS   str    
 6   SK_DPD                 int64  
 7   SK_DPD_DEF             int64  
dtypes: float64(2), int64(5), str(1)
memory usage: 610.4 MB
None

========== TYPES DES VARIABLES ==========
int64      5
float64    2
str        1
Name: count, dtype: int64

========== VARIABLES ==========
['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE', 'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF']

========== VALEURS MANQUANTES ==========


,Nombre_manquant,Pourcentage
CNT_INSTALMENT,26071,0.26
CNT_INSTALMENT_FUTURE,26087,0.26



========== APERCU ==========


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0



========== STATISTIQUES NUMERIQUES ==========


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,SK_DPD,SK_DPD_DEF
count,1.000136e+07,1.000136e+07,1.000136e+07,9.975287e+06,9.975271e+06,1.000136e+07,1.000136e+07
mean,1.903217e+06,2.784039e+05,-3.501259e+01,1.708965e+01,1.048384e+01,1.160693e+01,6.544684e-01
std,5.358465e+05,1.027637e+05,2.606657e+01,1.199506e+01,1.110906e+01,1.327140e+02,3.276249e+01
min,1.000001e+06,1.000010e+05,-9.600000e+01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.434405e+06,1.895500e+05,-5.400000e+01,1.000000e+01,3.000000e+00,0.000000e+00,0.000000e+00
50%,1.896565e+06,2.786540e+05,-2.800000e+01,1.200000e+01,7.000000e+00,0.000000e+00,0.000000e+00
75%,2.368963e+06,3.674290e+05,-1.300000e+01,2.400000e+01,1.400000e+01,0.000000e+00,0.000000e+00
max,2.843499e+06,4.562550e+05,-1.000000e+00,9.200000e+01,8.500000e+01,4.231000e+03,3.595000e+03


In [3]:
import pandas as pd
import numpy as np


# ============================================================
# 1) Chargement de la table
# ============================================================

pos_cash = pd.read_csv(
    "data/raw/POS_CASH_balance.csv"
)

print("========== DIMENSION INITIALE ==========")
print(pos_cash.shape)


# ============================================================
# 2) Traitement des valeurs manquantes
# ============================================================

print("\n========== VALEURS MANQUANTES AVANT ==========")

missing = pos_cash.isna().sum()
print(missing[missing > 0])


# Variables numériques
# Remplacement par la médiane

for col in [
    "CNT_INSTALMENT",
    "CNT_INSTALMENT_FUTURE"
]:
    
    pos_cash[col] = pos_cash[col].fillna(
        pos_cash[col].median()
    )


print("\nValeurs manquantes après traitement :")
print(pos_cash.isna().sum().sum())


# ============================================================
# 3) Création des variables comportementales
# Niveau crédit/client
# ============================================================

print("\nCréation des variables métier...")


# Variable retard
pos_cash["IS_DELAY"] = (
    pos_cash["SK_DPD"] > 0
).astype(int)


# Variable crédit terminé
pos_cash["IS_COMPLETED"] = (
    pos_cash["NAME_CONTRACT_STATUS"] == "Completed"
).astype(int)


# Variable crédit actif
pos_cash["IS_ACTIVE"] = (
    pos_cash["NAME_CONTRACT_STATUS"] == "Active"
).astype(int)



# ============================================================
# 4) Agrégation au niveau client
# ============================================================


pos_features = pos_cash.groupby(
    "SK_ID_CURR"
).agg(

    # Nombre total d'observations mensuelles
    NB_POS_RECORDS=(
        "SK_ID_PREV",
        "count"
    ),

    # Nombre de crédits différents
    NB_POS_CREDITS=(
        "SK_ID_PREV",
        "nunique"
    ),

    # Nombre de crédits actifs
    NB_ACTIVE_POS=(
        "IS_ACTIVE",
        "sum"
    ),

    # Nombre de crédits terminés
    NB_COMPLETED_POS=(
        "IS_COMPLETED",
        "sum"
    ),

    # Nombre de mois avec retard
    NB_DELAY_MONTHS=(
        "IS_DELAY",
        "sum"
    ),

    # Retard maximum observé
    MAX_DPD=(
        "SK_DPD",
        "max"
    ),

    # Retard corrigé maximum
    MAX_DPD_DEF=(
        "SK_DPD_DEF",
        "max"
    ),

    # Nombre moyen de mensualités
    AVG_CNT_INSTALMENT=(
        "CNT_INSTALMENT",
        "mean"
    ),

    # Nombre moyen de mensualités restantes
    AVG_CNT_INSTALMENT_FUTURE=(
        "CNT_INSTALMENT_FUTURE",
        "mean"
    )

).reset_index()



# ============================================================
# 5) Création des ratios comportementaux
# ============================================================


# Ratio de retard

pos_features["DPD_RATIO"] = (
    pos_features["NB_DELAY_MONTHS"]
    /
    pos_features["NB_POS_RECORDS"]
)



# Ratio de crédits actifs

pos_features["ACTIVE_RATIO"] = (
    pos_features["NB_ACTIVE_POS"]
    /
    pos_features["NB_POS_CREDITS"]
)



# Remplacement des valeurs infinies

pos_features.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)



# ============================================================
# 6) Traitement final des valeurs manquantes
# ============================================================


for col in pos_features.select_dtypes(
    include=np.number
).columns:

    if col != "SK_ID_CURR":

        pos_features[col] = pos_features[col].fillna(
            0
        )



# ============================================================
# 7) Vérification finale
# ============================================================


print("\n========== TABLE FINALE ==========")

print(pos_features.shape)


print("\nValeurs manquantes :")
print(
    pos_features.isna().sum().sum()
)


print("\nAperçu :")
display(
    pos_features.head()
)



# ============================================================
# 8) Sauvegarde
# ============================================================


pos_features.to_csv(
    "data/processed/pos_cash_features.csv",
    index=False
)


print(
    "\nFichier sauvegardé : pos_cash_features.csv"
)

========== DIMENSION INITIALE ==========
(10001358, 8)

========== VALEURS MANQUANTES AVANT ==========
CNT_INSTALMENT           26071
CNT_INSTALMENT_FUTURE    26087
dtype: int64

Valeurs manquantes après traitement :
0

Création des variables métier...

========== TABLE FINALE ==========
(337252, 12)

Valeurs manquantes :
0

Aperçu :


,SK_ID_CURR,NB_POS_RECORDS,NB_POS_CREDITS,NB_ACTIVE_POS,NB_COMPLETED_POS,NB_DELAY_MONTHS,MAX_DPD,MAX_DPD_DEF,AVG_CNT_INSTALMENT,AVG_CNT_INSTALMENT_FUTURE,DPD_RATIO,ACTIVE_RATIO
0,100001,9,2,7,2,1,7,7,4.000000,1.444444,0.111111,3.500000
1,100002,19,1,19,0,0,0,0,24.000000,15.000000,0.000000,19.000000
2,100003,28,3,26,2,0,0,0,10.107143,5.785714,0.000000,8.666667
3,100004,4,1,3,1,0,0,0,3.750000,2.250000,0.000000,3.000000
4,100005,11,1,9,1,0,0,0,11.727273,7.181818,0.000000,9.000000



Fichier sauvegardé : pos_cash_features.csv


# Transformation de la table POS_CASH_balance au niveau client

## 1. Objectif

La table `POS_CASH_balance` contient l'historique mensuel des crédits de type **POS (Point Of Sale)** et **Cash Loans** accordés aux clients.

Contrairement à la table `application_train` où une ligne correspond à un client, cette table contient plusieurs lignes pour un même client :

- un client peut avoir plusieurs crédits POS ;
- chaque crédit possède plusieurs observations mensuelles représentant son évolution dans le temps.

L'objectif est donc de transformer cette table afin d'obtenir :

- une seule ligne par client (`SK_ID_CURR`) ;
- des variables synthétiques représentant son comportement de remboursement ;
- des indicateurs exploitables par un modèle de Machine Learning de prédiction du risque de crédit.

---

# 2. Structure initiale de la table

Après chargement de la table :

- Nombre de lignes : **10 001 358**
- Nombre de variables : **8**

Chaque ligne représente un suivi mensuel d'un crédit POS.

Exemple :

| SK_ID_PREV | SK_ID_CURR | MONTHS_BALANCE | CNT_INSTALMENT | CNT_INSTALMENT_FUTURE | NAME_CONTRACT_STATUS | SK_DPD |
|------------|------------|----------------|----------------|-----------------------|----------------------|--------|
|1803195|182943|-31|48|45|Active|0|
|1715348|367990|-33|36|35|Active|0|
|1784872|397406|-32|12|9|Active|0|

Interprétation :

- `SK_ID_PREV` : identifiant du crédit précédent.
- `SK_ID_CURR` : identifiant du client.
- `MONTHS_BALANCE` : nombre de mois avant la demande actuelle.
- `CNT_INSTALMENT` : nombre total d'échéances prévues.
- `CNT_INSTALMENT_FUTURE` : nombre d'échéances restantes.
- `NAME_CONTRACT_STATUS` : état du contrat.
- `SK_DPD` : nombre de jours de retard de paiement.
- `SK_DPD_DEF` : nombre de jours de retard après prise en compte des défauts.

---

# 3. Analyse des valeurs manquantes

Avant traitement :

| Variable | Nombre de valeurs manquantes | Pourcentage |
|----------|------------------------------|-------------|
|CNT_INSTALMENT|26071|0.26%|
|CNT_INSTALMENT_FUTURE|26087|0.26%|

Ces valeurs manquantes représentent une faible proportion de la table.

Comme ces variables correspondent au nombre d'échéances prévues et restantes, leur absence signifie une information non disponible.

---

# 4. Traitement des valeurs manquantes

La méthode utilisée est l'imputation par la médiane.

Pourquoi la médiane ?

- Les variables financières peuvent contenir des valeurs extrêmes.
- La médiane est moins sensible aux valeurs aberrantes que la moyenne.
- Elle conserve une valeur représentative du comportement général des crédits.

Méthode appliquée :

- `CNT_INSTALMENT` → remplacement par la médiane.
- `CNT_INSTALMENT_FUTURE` → remplacement par la médiane.

Après traitement :


--------------------------------------------

Un hyperparamètre est un paramètre défini **avant l'entraînement** du modèle. Contrairement aux paramètres appris automatiquement par le modèle, les hyperparamètres contrôlent son fonctionnement et influencent directement ses performances.

Exemples d'hyperparamètres de XGBoost :

- **n_estimators** : nombre d'arbres construits.
- **max_depth** : profondeur maximale de chaque arbre.
- **learning_rate** : vitesse d'apprentissage du modèle.
- **subsample** : proportion des observations utilisée pour construire chaque arbre.
- **colsample_bytree** : proportion des variables utilisée pour chaque arbre.
- **gamma** : seuil minimal de réduction de perte nécessaire pour créer une nouvelle division.
- **min_child_weight** : nombre minimal d'observations dans une feuille.

---

## Comment les hyperparamètres sont-ils choisis ?

Les hyperparamètres sont recherchés automatiquement à l'aide de **GridSearchCV** (ou **RandomizedSearchCV**).

Le principe est le suivant :

1. Définir plusieurs valeurs possibles pour chaque hyperparamètre.
2. Générer toutes les combinaisons possibles.
3. Entraîner un modèle pour chaque combinaison.
4. Évaluer chaque modèle à l'aide d'une validation croisée (*Cross Validation*).
5. Comparer les performances selon une métrique (dans ce projet : **ROC-AUC**).
6. Conserver automatiquement la combinaison obtenant le meilleur score.

---

## Pourquoi utiliser la validation croisée ?

La validation croisée consiste à diviser les données en plusieurs sous-ensembles (*folds*).


Un hyperparamètre est un paramètre défini **avant l'entraînement** du modèle. Contrairement aux paramètres appris automatiquement par le modèle, les hyperparamètres contrôlent son fonctionnement et influencent directement ses performances.

## Exemples d'hyperparamètres de XGBoost :

- **n_estimators** : nombre d'arbres construits.
- **max_depth** : profondeur maximale de chaque arbre.
- **learning_rate** : vitesse d'apprentissage du modèle.
- **subsample** : proportion des observations utilisée pour construire chaque arbre.
- **colsample_bytree** : proportion des variables utilisée pour chaque arbre.
- **gamma** : seuil minimal de réduction de perte nécessaire pour créer une nouvelle division.
- **min_child_weight** : nombre minimal d'observations dans une feuille.

---

## Comment les hyperparamètres sont-ils choisis ?

Les hyperparamètres sont recherchés automatiquement à l'aide de **GridSearchCV** (ou **RandomizedSearchCV**).

Le principe est le suivant :

1. Définir plusieurs valeurs possibles pour chaque hyperparamètre.
2. Générer toutes les combinaisons possibles.
3. Entraîner un modèle pour chaque combinaison.
4. Évaluer chaque modèle à l'aide d'une validation croisée (*Cross Validation*).
5. Comparer les performances selon une métrique (dans ce projet : **ROC-AUC**).
6. Conserver automatiquement la combinaison obtenant le meilleur score.

---


